In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
data=pd.read_csv('TRAIN.csv')
data.head()

,F01,F02,F03,F04,F05,F06,F07,F08,F09,F10,...,F39,F40,F41,F42,F43,F44,F45,F46,F47,Class
0,0.185570,0.004568,0.005362,0.003335,0.005415,0.004895,0.012764,0.120138,0.140450,3.361753,...,0.041526,-0.230857,0.003310,0.042250,0.005975,0.002104,0.013878,0.001518,0.011347,0
1,0.369536,0.003983,0.003386,0.004902,0.007570,0.012136,0.118050,0.323925,0.132093,2.766117,...,-0.141285,-6.222857,0.834177,0.227968,0.018463,-0.020487,0.001246,0.007489,0.008724,1
2,0.602510,0.008442,0.012961,0.012870,0.046885,0.115401,0.065688,0.306677,0.498805,4.521201,...,0.011334,10.335251,-0.276614,-0.198900,-0.012756,0.014286,-0.001866,0.002687,0.013452,1
3,0.347957,0.064721,0.013611,0.011541,0.006492,0.008690,0.013192,0.164553,0.298665,3.170847,...,0.190479,2.864912,-1.921939,0.891690,1.108098,0.635431,0.081368,-0.000225,0.009166,0
4,0.233653,0.012217,0.010088,0.022095,0.026040,0.015062,0.016063,0.084648,0.213367,8.150943,...,0.203164,0.001812,-0.092731,0.005280,-0.213985,0.032195,0.002081,0.028930,-0.025912,1


In [3]:
X=data.drop('Class',axis=1)
y=data['Class']

In [4]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,train_size=0.25,random_state=42)

In [5]:
# from sklearn.preprocessing import StandardScaler
# scaler=StandardScaler()
# X_train_scale=scaler.fit_transform(X_train)
# X_test_scale=scaler.transform(X_test)

In [6]:
from sklearn.ensemble import RandomForestClassifier
clf=RandomForestClassifier(n_estimators=100,random_state=42,n_jobs=-1,class_weight='balanced')
clf.fit(X_train,y_train)

RandomForestClassifier(class_weight='balanced', n_jobs=-1, random_state=42)

In [7]:
y_prob=clf.predict_proba(X_test)[:,1]
y_pred=clf.predict(X_test)

In [8]:
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report,roc_auc_score

In [9]:
print(classification_report(y_test,y_pred))
print(confusion_matrix(y_test,y_pred))
print(accuracy_score(y_test,y_pred))
auc=roc_auc_score(y_test,y_prob)
print('auc socre',auc)

              precision    recall  f1-score   support

           0       0.95      0.99      0.97     19800
           1       0.99      0.92      0.95     13032

    accuracy                           0.96     32832
   macro avg       0.97      0.96      0.96     32832
weighted avg       0.97      0.96      0.96     32832

[[19695   105]
 [ 1073 11959]]
0.9641203703703703
auc socre 0.9963538624427207


In [10]:
print("Train Accuracy:",clf.score(X_train,y_train))
print("Test accuracy:",clf.score(X_test,y_test))

Train Accuracy: 1.0
Test accuracy: 0.9641203703703703


In [11]:
## checking the features importance.
feature_importance=pd.Series(clf.feature_importances_)
feature_importance.sort_values(ascending=False).head(10)

8     0.059939
18    0.058255
28    0.049845
0     0.044457
4     0.037972
24    0.036717
26    0.036078
25    0.032712
7     0.031620
27    0.029710
dtype: float64

In [12]:
## Lets do Cross Validation of our model
from sklearn.model_selection import cross_val_score

score=cross_val_score(clf,X,y,cv=5,scoring='roc_auc',n_jobs=-1)

In [13]:
print("CV scores:",score)
print("Mean CV AUC:",score.mean())

CV scores: [0.998885   0.99864426 0.99936353 0.9991356  0.99903094]
Mean CV AUC: 0.9990118665134154


In [15]:
## checking for Leakage or overfitting.
y_suff=np.random.permutation(y)

scores=cross_val_score(
    clf,
    X,
    y_suff,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1
)


In [16]:
print(scores.mean())

0.5019362190271854


In [17]:
## Hyper parameter Tuning
from sklearn.model_selection import RandomizedSearchCV

param_grid={
    'n_estimators':[200,300,500],
    'max_depth':[10,15,20,None],
    'min_samples_split':[2,5,10],
    'min_samples_leaf':[1,3,5,10],
    'max_features':['sqrt','log2',0.5]
}

In [18]:
random_search=RandomizedSearchCV(
    clf,
    param_distributions=param_grid,
    n_iter=25,
    scoring='roc_auc',
    cv=5,
    verbose=1,
    n_jobs=-1
)

In [20]:
random_search.fit(X_train,y_train)

Fitting 5 folds for each of 25 candidates, totalling 125 fits


RandomizedSearchCV(cv=5,
                   estimator=RandomForestClassifier(class_weight='balanced',
                                                    n_jobs=-1,
                                                    random_state=42),
                   n_iter=25, n_jobs=-1,
                   param_distributions={'max_depth': [10, 15, 20, None],
                                        'max_features': ['sqrt', 'log2', 0.5],
                                        'min_samples_leaf': [1, 3, 5, 10],
                                        'min_samples_split': [2, 5, 10],
                                        'n_estimators': [200, 300, 500]},
                   scoring='roc_auc', verbose=1)

In [21]:
print("Best Params:",random_search.best_params_)
print("Best CV AUC:",random_search.best_score_)

Best Params: {'n_estimators': 300, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': 20}
Best CV AUC: 0.9947695424224969


In [24]:
y_prob_hpt=clf.predict_proba(X_test)[:,1]
y_pred_hpt=random_search.predict(X_test)

In [25]:
print(classification_report(y_test,y_pred_hpt))
print(confusion_matrix(y_test,y_pred_hpt))
print(accuracy_score(y_test,y_pred_hpt))
auc=roc_auc_score(y_test,y_prob_hpt)
print('auc socre',auc)

              precision    recall  f1-score   support

           0       0.95      0.99      0.97     19800
           1       0.99      0.92      0.95     13032

    accuracy                           0.96     32832
   macro avg       0.97      0.96      0.96     32832
weighted avg       0.96      0.96      0.96     32832

[[19696   104]
 [ 1093 11939]]
0.9635416666666666
auc socre 0.9963538624427207
